##Passo 1: Importar o pandas e o quantstats

In [ ]:
import pandas as pd
import quantstats as qs 

# Passo 2: Leitura e tratamento inicial dos dados

In [ ]:
dados_empresas = pd.read_excel("statusinvest-busca-avancada.xlsx")

def get_base_ticker(ticker):
    return ''.join(filter(str.isalpha, ticker))


def get_priority(ticker):
    if ticker.endswith('4'):
        return 3
    elif ticker.endswith('11'):
        return 2
    elif ticker.endswith('3'):
        return 1
    else:
        return 0

dados_empresas['TICKER'] = dados_empresas['TICKER'].apply(get_base_ticker)

dados_empresas['Priority'] = dados_empresas['TICKER'].apply(get_priority)

dados_empresas = dados_empresas.sort_values(by=['TICKER', 'Priority'], ascending=[True, False])

dados_empresas = dados_empresas.drop_duplicates(subset='TICKER', keep='first')


In [ ]:

dados_empresas = dados_empresas[dados_empresas[' VALOR DE MERCADO'] > 1000000]
dados_empresas = dados_empresas[dados_empresas[' LIQUIDEZ MEDIA DIARIA'] > 500000]
dados_empresas = dados_empresas[dados_empresas['MARG. LIQUIDA'] >=0]
dados_empresas = dados_empresas[dados_empresas['MARGEM BRUTA'] >=0]
dados_empresas = dados_empresas[dados_empresas['MARGEM EBIT'] >=0]
dados_empresas = dados_empresas[dados_empresas['CAGR RECEITAS 5 ANOS'] >= 0]
dados_empresas = dados_empresas[dados_empresas['CAGR LUCROS 5 ANOS'] >= 0]
dados_empresas = dados_empresas[dados_empresas['ROE'] >= 0]
dados_empresas = dados_empresas[dados_empresas['LIQ. CORRENTE'] != 0]
dados_empresas = dados_empresas[dados_empresas[' PEG Ratio'] > 0]
dados_empresas = dados_empresas[dados_empresas['P/VP'] <= 2]
dados_empresas.fillna(0, inplace=True)

# Passo 3: Criar o ranking dos indicadores.

In [ ]:
dados_empresas['ranking_margem_liq'] = dados_empresas['MARG. LIQUIDA'].rank(ascending = False)

dados_empresas['ranking_pl'] = dados_empresas['P/L'].rank(ascending = True)

dados_empresas['ranking_psr'] = dados_empresas['PSR'].rank(ascending = False)

dados_empresas['ranking_roe'] = dados_empresas['ROE'].rank(ascending = False)

dados_empresas['ranking_p_ebit'] = dados_empresas['P/EBIT'].rank(ascending = True)

dados_empresas['ranking_lpa'] = dados_empresas[' LPA'].rank(ascending = False)

dados_empresas['ranking_liq_corr'] = dados_empresas['LIQ. CORRENTE'].rank(ascending = False)

dados_empresas['ranking_mktValue'] = dados_empresas[' VALOR DE MERCADO'].rank(ascending = False)

In [ ]:
dados_empresas['ranking_final'] = dados_empresas['ranking_margem_liq'] + dados_empresas['ranking_psr'] + dados_empresas['ranking_roe'] + dados_empresas['ranking_liq_corr'] + dados_empresas['ranking_lpa'] + dados_empresas['ranking_p_ebit'] + dados_empresas['ranking_pl'] + dados_empresas['ranking_mktValue']

dados_empresas['ranking_final'] = dados_empresas['ranking_final'].rank()

In [ ]:
dados_empresas.sort_values('ranking_final').head(10)

# Passo 6: Criar  as carteiras. 

In [ ]:
dados_empresas = dados_empresas[dados_empresas['ranking_final'] <= 10]
dados_empresas = dados_empresas["TICKER"]
dados_empresas